In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
import gradio as gd

In [ ]:
filename = "the_nestle_hr_policy_pdf_2012.pdf"
loader = PyPDFLoader(filename)
docs = loader.load()
docs

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 50)
chunks = text_splitter.split_documents(docs)
len(chunks)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model = "models/gemini-embedding-001", google_api_key = "YOUR_GOOGLE_API_KEY")

In [ ]:
vectorstoredb = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vectorstoredb.as_retriever(search_type = "similarity", search_kwargs = {"k" : 4})

In [ ]:
prompt = ChatPromptTemplate.from_template(
    '''
    You are a helpful Assistant,
    Answer only on the basis of the provided transcript,
    If you don't know the answer just say you don't know the answer.
    {context}
    Question : {query}
'''
)

In [ ]:
llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash", google_api_key = "YOUR_GOOGLE_API_KEY")

In [ ]:
parser = StrOutputParser()

In [ ]:
def score_calc(context_docs, answer):
    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer = True)
    scores = scorer.score(context_docs, answer)
    for metric, result in scores.items():
        src = f"{metric} : Precision : {result.precision:.2f}"
    return src

In [ ]:
def chatbot(query):
    retrieved_docs = retriever.invoke(query)
    context_docs = "\n\n".join(doc.page_content for doc in retrieved_docs)
    chain = prompt | llm | parser
    answer = chain.invoke({"context": context_docs, "query" : query})
    src = score_calc(context_docs, answer)
    return answer + "\n" + src

In [ ]:
# For installing gradio use command %pip install gradio 

iface = gd.Interface(fn = chatbot, inputs = gd.components.Textbox(lines = 7, label = "Enter your Text :"), outputs= gd.components.Textbox(lines = 7, label = "Output"))
iface.launch()